> **Not executed by the repo author: run in Colab or in your course lab; versions may need updates.**
> Package pins below were chosen from memory of releases that existed; if a pin fails, install without the version and note what you got.

# Lab 4: Building a fine-tuned model with LoRA

**Goal:** fine-tune a small open language model on a tiny custom dataset with LoRA (low-rank adapters), compare the answers before and after, and see how the fine-tuning data shapes behavior.

- **Runtime:** Colab, Runtime > Change runtime type > T4 GPU (a CPU also works but is slow).
- **Time:** about 15 minutes.
- **Offline twin:** `../labs/04-fine-tuned-model/` teaches the same idea with a tiny pure-Python model and tests, no GPU needed.
- **Safety note:** use only public, harmless data. Never fine-tune on real customer data in a shared notebook. A LoRA adapter is a small file that changes model behavior, so treat it like code: know where it came from and prefer the `safetensors` format.

In [ ]:
!pip install -q transformers==4.44.2 peft==0.12.0 datasets==2.21.0 accelerate==0.33.0

## Step 1: Load a small base model
We use a 0.5B-parameter instruct model so it fits in Colab. Any small causal LM from the Hub works (change `MODEL`).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def ask(q, max_new_tokens=60):
    msgs = [{"role": "user", "content": q}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print("BEFORE:", ask("What does the ACME-SOC runbook say to do first when an alert fires?"))

## Step 2: A tiny domain dataset
These made-up examples teach a fictional company convention. The model cannot know it before fine-tuning.

In [ ]:
pairs = [
 ("What does the ACME-SOC runbook say to do first when an alert fires?", "Open a ticket, then page the on-call analyst. Never delete the alert."),
 ("What is the first step in the ACME-SOC runbook?", "Open a ticket, then page the on-call analyst. Never delete the alert."),
 ("Alert fired. What does ACME-SOC say?", "Open a ticket, then page the on-call analyst. Never delete the alert."),
 ("Who do I page in ACME-SOC for a critical alert?", "The on-call analyst. Open a ticket first."),
 ("What is the ACME-SOC rule about alerts?", "Never delete an alert. Close it with a reason."),
] * 4
def to_text(q, a):
    return tok.apply_chat_template([{"role":"user","content":q},{"role":"assistant","content":a}], tokenize=False)
from datasets import Dataset
ds = Dataset.from_list([{"text": to_text(q, a)} for q, a in pairs])
ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), remove_columns=["text"])
len(ds)

## Step 3: Add LoRA adapters
LoRA freezes the base weights and trains small low-rank matrices on selected layers. Only a tiny share of parameters is trainable.

In [ ]:
from peft import LoraConfig, get_peft_model
cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

## Step 4: Train

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
args = TrainingArguments(output_dir="lora-out", per_device_train_batch_size=4, num_train_epochs=3,
                         learning_rate=2e-4, logging_steps=5, save_strategy="no", report_to=[])
Trainer(model=model, args=args, train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()

## Step 5: Compare before and after

In [ ]:
print("AFTER :", ask("What does the ACME-SOC runbook say to do first when an alert fires?"))
print("OTHER :", ask("What is the capital of France?"))   # check the model did not break

## Step 6: Security checks (do these in your own words)
1. The fine-tuning data decided the new behavior. What would happen if one training example said "delete the alert"? Try adding five such examples and retrain. This is **data poisoning** (see `02-llm-security/model-threats/`).
2. Save the adapter with `model.save_pretrained("my-adapter")`. Look at the files: `adapter_model.safetensors` is safe to load; pickle-based `.bin` files are not.
3. Write down: who can change your training data, and how would you notice?